In [0]:
dbutils.widgets.text("test_library_ref", "", "Test Library Ref")

test_library_ref = dbutils.widgets.get("test_library_ref")

if test_library_ref != "":
    # If the value looks like a version number (starts with digit), prepend package name
    if test_library_ref[0].isdigit():
        %pip install 'databricks-labs-dqx=={test_library_ref}'
    else:
        %pip install '{test_library_ref}'
else:
    %pip install databricks-labs-dqx

%restart_python

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.checks_serializer import ChecksNormalizer
from databricks.sdk import WorkspaceClient
import os
import yaml

In [0]:
dbutils.widgets.text("table", "", "Table Name")
dbutils.widgets.text("catalog", "lakehouse", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")

table = dbutils.widgets.get("table")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
input_df = spark.sql(f"SELECT * FROM {catalog}.{schema}.{table}")
input_df.display()

In [0]:
ws = WorkspaceClient()

input_df = spark.sql(f"SELECT * FROM {catalog}.{schema}.{table}")

profiler = DQProfiler(ws)

summary_stats, profiles = profiler.profile(
    input_df, 
    options={"sample_fraction": 0.5}
)

print(yaml.safe_dump(summary_stats))

In [0]:
print(profiles)

In [0]:
generator = DQGenerator(ws)
checks = generator.generate_dq_rules(profiles)
print(yaml.safe_dump(ChecksNormalizer.normalize(checks)))